# Train the Model

In [ ]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import grain.python as pygrain
import tiktoken
import optax # for optimizers

In [ ]:
from src.model import NanoLLM
from src.data_loader import load_text_from_file, preprocess_data
from src.inference import generate_text
from src.config import (
    TokenizerConfig,
    model_config,
    training_config,
    TINYSTORIES_FILE,
    CHECKPOINTS_DIR
)

## Setup

### Understand characteristics of the dataset

In [ ]:
# Define delimiter and tokenizer name explicitly for TinyStories dataset
DELIMITER = "<|endoftext|>"
TOKENIZER_NAME = "gpt2"

# Get configurations
tokenizer_config = TokenizerConfig(delimiter=DELIMITER, name=TOKENIZER_NAME)
max_paragraphs = training_config.max_stories
maxlen = model_config.maxlen

In [ ]:
# Load stories (includes validation of the path)
stories = load_text_from_file(TINYSTORIES_FILE, DELIMITER, max_paragraphs)

# Pre-process the data and calculate batch count
dataloader, batches_per_epoch = preprocess_data(
    list_of_paragraphs=stories,
    batch_size=training_config.batch_size,
    maxlen=model_config.maxlen,
    delimiter=DELIMITER,
    num_epochs=training_config.num_epochs,
    shuffle=training_config.shuffle,
    seed=training_config.seed
)

In [ ]:
# Calculate training steps
total_steps, warmup_steps = training_config.calculate_training_steps(batches_per_epoch)
print(f"Total training steps: {total_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")

### Define the training step: model, loss function, and learning rate schedule

In [ ]:
# Define model
model = NanoLLM()

In [ ]:
# Define loss function
def loss_fn(model, batch):
    inputs, targets = batch

    # Feed the inputs to the model. The results (logits) represent the denormalized likelihoods for each word in the dictionary that that word is next.
    logits = model(inputs)

    # Convert to probabilities (likely normalized)
    loss = optax.softmax_cross_entropy_with_integer_labels(
        logits, targets
    ).mean()

    # return both the probabilities and the original results
    return loss, logits

In [ ]:
# Create learning rate schedule using config values
learning_rate_schedule = optax.warmup_cosine_decay_schedule(
    init_value=training_config.lr_init_value,
    peak_value=training_config.lr_peak_value,
    warmup_steps=warmup_steps,
    decay_steps=total_steps,
    end_value=training_config.lr_end_value
)

### Define an optimizer to control how the network updates

In [ ]:
optimizer = nnx.ModelAndOptimizer(
    model,
    optax.adamw(
        learning_rate=learning_rate_schedule,
        weight_decay=training_config.weight_decay
    )
)

In [ ]:
# During training, we want to keep track of the average of the loss function: metrics.Average('loss')
metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average('loss'),
)

### Define the training step method (JIT-compile)

In [ ]:
@nnx.jit
def train_step(model, optimizer, metrics, batch):
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(model, batch)

    metrics.update(loss=loss, logits=logits, labels=batch[1])
    optimizer.update(grads)

## Run the training loop

In [ ]:
metrics_history = {'train_loss': []}

# slides the inputs over by one index so we are always comparing the inputs to the next token (the target)
prep_target_batch = jax.vmap(
    lambda tokens: jnp.concatenate((tokens[1:], jnp.array([0]))))

for epoch in range(training_config.num_epochs):
    step = 0

    # pull a batch from the dataloader
    for batch in dataloader:
        input_batch = jnp.array(jnp.array(batch).T).astype(jnp.int32)
        target_batch = prep_target_batch(
            jnp.array(jnp.array(batch).T)).astype(jnp.int32)
        print(".", end="")

        # training step!
        train_step(model, optimizer, metrics, (input_batch, target_batch))

        if (step + 1) % training_config.log_every_n_steps == 0:
            for metric, value in metrics.compute().items():
                # update metrics history so we can track how well the model is doing
                metrics_history[f'train_{metric}'].append(value)
            metrics.reset()

            current_lr = learning_rate_schedule(step)
            print(f"\nEpoch: {epoch + 1}, Step {step + 1}, Loss: {metrics_history['train_loss'][-1]:.4f}, "
                  f"LR: {current_lr:.2e}")
        step += 1

In [ ]:
# Visualize the training
import matplotlib.pyplot as plt
plt.plot(metrics_history['train_loss'])
plt.title('Training Loss — 3 steps, 100 stories')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.show()

## Save model checkpoints

In [ ]:
from pathlib import Path
import orbax
from src.config import format_path_for_display

# Use centralized checkpoint directory
checkpoint_path = CHECKPOINTS_DIR / "nano_checkpoint.orbax"

checkpointer = orbax.checkpoint.PyTreeCheckpointer()

checkpointer.save(checkpoint_path, nnx.state(model), force=True)
print(f"Model saved as {format_path_for_display(checkpoint_path)}")